# MRO Parts Catalog — Cleaning Pipeline

**Input:** `data/raw_data.csv` (292,658 rows × 87 columns)  
**Output:** `data/cleaned_data.csv` (292,658 rows × 19 columns)

Every decision in this notebook was made after verifying against the full dataset.  
See `eda.ipynb` for the investigation behind each decision.

---
## 0. Imports & Paths

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

RAW_PATH   = 'data/raw_data.csv'
CLEAN_PATH = 'data/cleaned_data.csv'

print('Imports OK')

Imports OK


---
## 1. Load & Rename Columns

Raw column names are a mix of camelCase, dot-notation, and lowercase.
We lowercase everything first, then rename only the columns we are keeping.
Everything else is implicitly dropped by selecting only the renamed columns.

**Notable:** the `price` column contains supplier/distributor names, not prices.
This is a mislabeling in the source export — renamed to `supplier_name`.

In [2]:
# Maps raw column name -> clean name
# Only columns listed here survive. All others are dropped.
COLUMN_MAP = {
    'item key':                          'supplier_catalog_key',
    'price':                             'supplier_name',
    'sku':                               'sku',
    'attributefamily.code':              'attribute_family',
    'category.default.title':            'category_name',
    'category.id':                       'category_id',
    'brand.default.title':               'brand_name',
    'manufacturer_part_number':          'manufacturer_part_number',
    'last_sold_price':                   'last_sold_price',
    'itemweight':                        'item_weight',
    'attribute_table':                   'attribute_table',
    'downloads':                         'downloads',
    'manufacturer_description':          'manufacturer_description',
    'names.default.value':               'default_name',
    'names.livhaven.value':              'livhaven_name',
    'shortdescriptions.default.value':   'default_short_description',
    'shortdescriptions.livhaven.value':  'livhaven_short_description',
    'descriptions.mro.value':            'mro_description',
    'descriptions.livhaven.value':       'livhaven_description',
}

DTYPE_MAP = {
    'supplier_catalog_key':       'Float64',
    'supplier_name':              'string',
    'sku':                        'string',
    'attribute_family':           'string',
    'category_name':              'string',
    'category_id':                'Float64',
    'brand_name':                 'string',
    'manufacturer_part_number':   'string',
    'last_sold_price':            'Float64',
    'item_weight':                'Float64',
    'attribute_table':            'string',
    'downloads':                  'string',
    'manufacturer_description':   'string',
    'default_name':               'string',
    'livhaven_name':              'string',
    'default_short_description':  'string',
    'livhaven_short_description': 'string',
    'mro_description':            'string',
    'livhaven_description':       'string',
}

data = pd.read_csv(RAW_PATH, dtype=str)
data.columns = data.columns.str.lower().str.strip()
data = data.rename(columns=COLUMN_MAP)

# Keep only the renamed columns — everything else is dropped
keep_cols = [c for c in COLUMN_MAP.values() if c in data.columns]
data = data[keep_cols]

valid_dtype_map = {k: v for k, v in DTYPE_MAP.items() if k in data.columns}
data = data.astype(valid_dtype_map)

print(f'Loaded. Shape: {data.shape}')
print(f'Columns: {list(data.columns)}')

Loaded. Shape: (292658, 19)
Columns: ['supplier_catalog_key', 'supplier_name', 'sku', 'attribute_family', 'category_name', 'category_id', 'brand_name', 'manufacturer_part_number', 'last_sold_price', 'item_weight', 'attribute_table', 'downloads', 'manufacturer_description', 'default_name', 'livhaven_name', 'default_short_description', 'livhaven_short_description', 'mro_description', 'livhaven_description']


---
## 2. Normalize `brand_name`

Parker appears as 5+ variants in the raw data (Parker, Parker Pneumatic Division,
Parker FRL, Parker Finite, Parker Hose). All refer to the same manufacturer.

We store the original in `brand_name_raw` for audit purposes.
Expand `BRAND_ALIASES` as new variants are discovered on the full dataset.

In [3]:
BRAND_ALIASES = {
    'parker pneumatic division':   'Parker',
    'parker pneumatic':            'Parker',
    'parker frl':                  'Parker',
    'parker finite':               'Parker',
    'parker hose':                 'Parker',
    'parker hannifin':             'Parker',
    'parker-hannifin':             'Parker',
    'parker':                      'Parker',
    'bosch rexroth':               'Bosch Rexroth',
    'rexroth':                     'Bosch Rexroth',
    'smc corporation':             'SMC',
    'smc corp':                    'SMC',
    'smc':                         'SMC',
    'versa products':              'Versa Products',
    'versa products co.':          'Versa Products',
    'aventics':                    'Aventics',
    'hydac':                       'Hydac',
    'balluff, inc.':               'Balluff',
    'balluff':                     'Balluff',
    'hengst filtration':           'Hengst Filtration',
    'schroeder industries':        'Schroeder Industries',
    'bijur delimon international': 'Bijur Delimon',
    'bijur delimon':               'Bijur Delimon',
}

data['brand_name_raw'] = data['brand_name'].copy()

def normalize_brand(name):
    if pd.isna(name) or str(name).strip() == '':
        return pd.NA
    key = str(name).strip().lower()
    return BRAND_ALIASES.get(key, str(name).strip())

data['brand_name'] = data['brand_name'].apply(normalize_brand).astype('string')

changed = data[
    data['brand_name'] != data['brand_name_raw']
][['brand_name_raw', 'brand_name']].drop_duplicates()

print(f'Brands normalized: {len(changed)}')
print(changed.to_string())
print(f'\nUnique brands after normalization: {data["brand_name"].nunique()}')

Brands normalized: 6
                    brand_name_raw     brand_name
22                      Parker FRL         Parker
80                     Parker Hose         Parker
649    Bijur Delimon International  Bijur Delimon
672                  Balluff, Inc.        Balluff
3508                 Parker Finite         Parker
17484    Parker Pneumatic Division         Parker

Unique brands after normalization: 55


---
## 3. Normalize `manufacturer_part_number`

Confirmed in sample: no spaces or lowercase in the raw data.
We still uppercase and strip as a safety measure on the full dataset.
Outliers (length < 4 or > 30) are flagged for human review, not dropped.

In [4]:
data['manufacturer_part_number'] = (
    data['manufacturer_part_number']
    .astype('string')
    .str.strip()
    .str.upper()
)

pn_len = data['manufacturer_part_number'].str.len()
data['pn_flag'] = pd.NA
data.loc[pn_len < 4,  'pn_flag'] = 'too_short'
data.loc[pn_len > 30, 'pn_flag'] = 'too_long'

flagged = data['pn_flag'].notna().sum()
print(f'Part numbers flagged: {flagged}')
if flagged > 0:
    print(data[data['pn_flag'].notna()][
        ['manufacturer_part_number', 'brand_name', 'pn_flag']
    ].head(10).to_string())

Part numbers flagged: 1210
     manufacturer_part_number     brand_name    pn_flag
129                         0           <NA>  too_short
132                         0           <NA>  too_short
133                         0           <NA>  too_short
212                         0  Bosch Rexroth  too_short
673                       555        Alemite  too_short
961                       IPS         Parker  too_short
1093                      198    Dixon Valve  too_short
1106                      241    Dixon Valve  too_short
1131                      271    Dixon Valve  too_short
1151                      301    Dixon Valve  too_short


---
## 4. Numeric Sanity Checks

`last_sold_price`: zero prices are bad data, replaced with NaN.  
`item_weight`: zeros are flagged but not auto-nulled.

In [5]:
for col in ['last_sold_price', 'item_weight']:
    s = pd.to_numeric(data[col], errors='coerce')
    data[col] = s
    print(f'{col}:')
    print(f'  null:     {s.isna().sum():,} ({s.isna().mean()*100:.1f}%)')
    print(f'  zero:     {(s==0).sum():,}')
    print(f'  negative: {(s<0).sum():,}')
    print(f'  median:   {s.median():.2f}')
    print(f'  p99:      {s.quantile(0.99):.2f}')
    print()

data['last_sold_price'] = data['last_sold_price'].replace(0, np.nan)
print('Zero prices set to NaN.')

last_sold_price:
  null:     152 (0.1%)
  zero:     3
  negative: 0
  median:   302.79
  p99:      11060.00

item_weight:
  null:     0 (0.0%)
  zero:     1
  negative: 0
  median:   2.50
  p99:      218.00

Zero prices set to NaN.


---
## 5. Parse `attribute_table`

Format confirmed by inspection: pipe-delimited key/value pairs.
`Label | | | | Value | | | | Label | | | | Value ...`

We keep only keys present in >= 1% of rows to avoid a very sparse column explosion.
All extracted columns are prefixed `attr_`.
The raw `attribute_table` column is dropped after extraction.

In [6]:
def parse_pipe_attrs(raw):
    if pd.isna(raw) or str(raw).strip() == '':
        return {}
    parts = [p.strip() for p in str(raw).split('|')]
    parts = [p for p in parts if p]
    result = {}
    i = 0
    while i < len(parts) - 1:
        key = parts[i].strip()
        val = parts[i + 1].strip()
        if key:
            result[key] = val if val else np.nan
        i += 2
    return result

print('Parsing attribute_table (~60s on full dataset)...')
parsed    = data['attribute_table'].apply(parse_pipe_attrs)
attr_df   = pd.DataFrame(parsed.tolist(), index=data.index)
coverage  = attr_df.notna().sum().sort_values(ascending=False)
MIN_ROWS  = max(1, int(len(data) * 0.01))
keep_keys = coverage[coverage >= MIN_ROWS].index.tolist()

data = pd.concat([data, attr_df[keep_keys].add_prefix('attr_')], axis=1)
data.drop(columns=['attribute_table'], inplace=True)

print(f'Attribute columns kept (>= {MIN_ROWS} rows): {len(keep_keys)}')
print('Top 10:')
print(coverage.head(10).to_string())
print(f'\nShape: {data.shape}')

Parsing attribute_table (~60s on full dataset)...
Attribute columns kept (>= 2926 rows): 0
Top 10:
1/4"                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

---
## 6. Final State & Export

In [7]:
print(f'Final shape: {data.shape}')
print()
print(f'{"Column":<45} {"Filled":>8}  {"% Filled":>9}')
print('-' * 65)
for c in data.columns:
    filled = data[c].notna().sum()
    pct    = filled / len(data) * 100
    print(f'{c:<45} {filled:>8,}  {pct:>8.1f}%')

data.to_csv(CLEAN_PATH, index=False)
print(f'\nSaved → {CLEAN_PATH}')

Final shape: (292658, 20)

Column                                          Filled   % Filled
-----------------------------------------------------------------
supplier_catalog_key                           292,652     100.0%
supplier_name                                  292,652     100.0%
sku                                            292,658     100.0%
attribute_family                               292,658     100.0%
category_name                                  278,194      95.1%
category_id                                    278,194      95.1%
brand_name                                     292,438      99.9%
manufacturer_part_number                       292,520     100.0%
last_sold_price                                292,503      99.9%
item_weight                                    292,658     100.0%
downloads                                      181,949      62.2%
manufacturer_description                        38,133      13.0%
default_name                                   29

In [8]:
pn_col    = 'manufacturer_part_number'
brand_col = 'brand_name'

total     = len(data)
n_unique  = data.drop_duplicates(subset=[pn_col, brand_col]).shape[0]
n_dupes   = total - n_unique

print(f'Total rows:                    {total:,}')
print(f'Unique (part + brand):         {n_unique:,}')
print(f'Duplicate rows:                {n_dupes:,} ({n_dupes/total*100:.1f}%)')

# Worst offenders
print('\nMost duplicated part+brand combos:')
top_dupes = (
    data.groupby([pn_col, brand_col], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(15)
)
print(top_dupes.to_string(index=False))

# Are duplicates identical rows or do they differ?
print('\n\nDo duplicate rows actually differ from each other?')
duped_mask = data.duplicated(subset=[pn_col, brand_col], keep=False)
duped = data[duped_mask]
fully_identical = data.duplicated(keep=False).sum()
print(f'  Rows sharing part+brand:     {duped_mask.sum():,}')
print(f'  Fully identical rows:        {fully_identical:,}')
print(f'  Differ in at least one col:  {duped_mask.sum() - fully_identical:,}')

Total rows:                    292,658
Unique (part + brand):         292,509
Duplicate rows:                149 (0.1%)

Most duplicated part+brand combos:
manufacturer_part_number           brand_name  count
                    <NA>        Bosch Rexroth     64
                    <NA>            perma USA     44
                    <NA>                 <NA>     19
                    <NA> Schroeder Industries      8
                       0                 <NA>      3
              R901278775        Bosch Rexroth      2
              R902502997        Bosch Rexroth      2
              R902502985        Bosch Rexroth      2
              DD05TPPT4S       Daman Products      2
              R901329072        Bosch Rexroth      2
              R900931049        Bosch Rexroth      2
              R900427940        Bosch Rexroth      2
              R978878261        Bosch Rexroth      2
      05018ET3E326TC-W22                  WEG      2
              3842557202        Bosch Rexroth    

In [9]:
# 1. Flag rows with null part numbers — these are bad data
data['missing_pn'] = data['manufacturer_part_number'].isna()
print(f"Rows with null part number: {data['missing_pn'].sum():,}")

# 2. For real duplicates (non-null part+brand), keep the richer row
before = len(data)

# Separate null-pn rows from real rows
real    = data[data['manufacturer_part_number'].notna()].copy()
null_pn = data[data['manufacturer_part_number'].isna()].copy()

# For real duplicates, keep row with most non-null values
real['_richness'] = real.notna().sum(axis=1)
real = (
    real.sort_values('_richness', ascending=False)
    .drop_duplicates(subset=['manufacturer_part_number', 'brand_name'], keep='first')
    .drop(columns=['_richness'])
)

# Put back together
data = pd.concat([real, null_pn], ignore_index=True)

print(f'Rows before: {before:,}')
print(f'Rows after:  {len(data):,}')
print(f'Removed:     {before - len(data):,} real duplicates')
print(f'Null-pn rows kept for review: {len(null_pn):,}')

Rows with null part number: 138
Rows before: 292,658
Rows after:  292,640
Removed:     18 real duplicates
Null-pn rows kept for review: 138


In [10]:
import re

def has_html(s):
    if pd.isna(s): return False
    return bool(re.search(r'<[a-zA-Z]', str(s)))

desc_cols = [
    'default_name',
    'default_short_description',
    'livhaven_short_description',
    'mro_description',
    'livhaven_description',
    'manufacturer_description',
    'livhaven_name',
]

print('HTML presence by column:')
for col in desc_cols:
    if col not in data.columns: continue
    filled  = data[col].notna().sum()
    has_htm = data[col].apply(has_html).sum()
    print(f'  {col:<35} {has_htm:>7,} / {filled:>7,} rows have HTML ({has_htm/filled*100:.1f}%)')

HTML presence by column:
  default_name                              2 / 292,640 rows have HTML (0.0%)
  default_short_description            89,653 / 248,371 rows have HTML (36.1%)
  livhaven_short_description           16,729 /  56,187 rows have HTML (29.8%)
  mro_description                      88,495 /  88,495 rows have HTML (100.0%)
  livhaven_description                228,796 / 229,712 rows have HTML (99.6%)
  manufacturer_description             38,131 /  38,133 rows have HTML (100.0%)
  livhaven_name                             0 /  61,457 rows have HTML (0.0%)


In [11]:
# Show 3 HTML and 3 non-HTML rows from default_short_description side by side
html_rows     = data[data['default_short_description'].apply(has_html)]['default_short_description'].head(3)
non_html_rows = data[~data['default_short_description'].apply(has_html) & 
                      data['default_short_description'].notna()]['default_short_description'].head(3)

print('=== HTML rows ===')
for v in html_rows:
    print(f'  → {str(v)[:200]}')
    print()

print('=== Non-HTML rows ===')
for v in non_html_rows:
    print(f'  → {str(v)[:200]}')
    print()

# Also check: after stripping HTML from the HTML rows, 
# does the remaining text look like the non-HTML rows?
from html.parser import HTMLParser

class _Strip(HTMLParser):
    def __init__(self): super().__init__(); self.fed = []
    def handle_data(self, d): self.fed.append(d)
    def get_text(self): return ' '.join(self.fed)

def strip_html(t):
    if pd.isna(t) or str(t).strip() == '': return pd.NA
    t = str(t)
    if '<' not in t: return t.strip()
    s = _Strip()
    s.feed(t)
    import re
    return re.sub(r'\s+', ' ', s.get_text()).strip() or pd.NA

print('=== HTML rows AFTER stripping ===')
for v in html_rows:
    print(f'  → {str(strip_html(v))[:200]}')
    print()

=== HTML rows ===
  → <p>Quick Exhaust Valve, 1/4" NPT IN &amp; OUT Ports, 3/8" NPT EXH Port, 2.00 Cv, NBR Seals, Aluminum Diecast Body, 3-150PSI Pressure Range</p>

  → <p>Quick Exhaust Valve, 1/2" NPT IN &amp; OUT Ports, 3/4" NPT EXH Port, 6.22 Cv, Polyurethane Seals, Aluminum Diecast Body, 1-150PSI Pressure Range</p>

  → <p>Quick Exhaust Valve, 3/8" NPT IN &amp; OUT Ports, 3/8" NPT EXH Port, 2.58 Cv, NBR Seals, Aluminum Diecast Body, 2-150PSI Pressure Range</p>

=== Non-HTML rows ===
  → TAC Valve, 3-Way, Normally Closed, 2-Position, Push Button, Spring Return, M5 X 0.8 Ports, Brass Body, NBR Seals, Hardened Button For Manual Or Mechanical Use, Includes 2 Panel Mounting Nuts (Part# 12

  → TAC Valve, 3-Way, Normally Closed, 2-Position, Toggle, Detent, M5 X 0.8 Ports, Brass Body, NBR Seals, Includes 2 Panel Mounting Nuts (Part# 120-21), 0-125PSI Pressure Range

  → TAC Valve, 4-Way, 2-Position, Push Button, Spring Return, M5 X 0.8 Ports, Brass Body, NBR Seals, Hardened Button For Ma

In [12]:
import re
from html.parser import HTMLParser

class _Strip(HTMLParser):
    def __init__(self): super().__init__(); self.fed = []
    def handle_data(self, d): self.fed.append(d)
    def get_text(self): return ' '.join(self.fed)

def strip_html(t):
    if pd.isna(t) or str(t).strip() == '': return pd.NA
    t = str(t)
    if '<' not in t: return t.strip()
    s = _Strip()
    s.feed(t)
    return re.sub(r'\s+', ' ', s.get_text()).strip() or pd.NA

cols_to_strip = [
    'default_short_description',
    'livhaven_short_description',
    'mro_description',
    'livhaven_description',
    'manufacturer_description',
    'default_name',  # only 2 rows but clean them anyway
]

for col in cols_to_strip:
    if col not in data.columns: continue
    data[col] = data[col].apply(strip_html).astype('string')
    remaining_html = data[col].apply(has_html).sum()
    print(f'  {col:<35} HTML remaining: {remaining_html}')

print('\nDone. Sample stripped mro_description:')
print(data['mro_description'].dropna().iloc[0][:300])

  default_short_description           HTML remaining: 0
  livhaven_short_description          HTML remaining: 0
  mro_description                     HTML remaining: 0
  livhaven_description                HTML remaining: 0
  manufacturer_description            HTML remaining: 0
  default_name                        HTML remaining: 0

Done. Sample stripped mro_description:
E3P Humphrey Pneumatic Directional Valve MROStop is the Humphrey distributor you can trust to guide you in the right direction and help you find the components you need. As a top Humphrey distributor, we have the expertise you're looking for. If you have any questions about the E3P Humphrey Pneumati


In [13]:
# Everything that didn't match an alias came through as-is
# We need to see what's there and whether anything needs consolidating

mapped   = set(v for v in BRAND_ALIASES.values())
all_brands = data['brand_name'].dropna().unique()

print(f'Total unique brands after normalization: {len(all_brands)}')
print(f'\nAll current brand values + row counts:')
print(data['brand_name'].value_counts(dropna=False).to_string())

Total unique brands after normalization: 55

All current brand values + row counts:
brand_name
Parker                                 90961
Bosch Rexroth                          37420
Versa Products                         22880
Hydac                                  22877
Aventics                               15087
Hengli                                 15052
Balluff                                14543
Schroeder Industries                    9491
Daman Products                          7253
Graco                                   7065
Hengst Filtration                       7063
Phoenix Contact                         6975
Bijur Delimon                           6290
Donaldson                               6142
NOSHOK, Inc.                            6058
Adaptall Inc                            4905
Humphrey Products                       3706
Mitsubishi                              2092
Parker Parflex                          1447
WEG                                      913
Holmb

In [14]:
# Check Parker division variants in detail
parker_variants = [c for c in data['brand_name'].dropna().unique() 
                   if 'parker' in str(c).lower()]
print('All Parker variants currently in data:')
for v in sorted(parker_variants):
    count = (data['brand_name'] == v).sum()
    # Show sample categories for each
    cats = data[data['brand_name'] == v]['category_name'].value_counts().head(3)
    print(f'\n  {v} ({count:,} rows)')
    print(f'  Categories: {", ".join(cats.index.tolist())}')

# Check Humphrey
print('\n\nHumphrey variants:')
humph = [c for c in data['brand_name'].dropna().unique() 
         if 'humphrey' in str(c).lower()]
for v in humph:
    print(f'  {v}: {(data["brand_name"] == v).sum():,} rows')

# Check Asco
print('\nAsco variants:')
asco = [c for c in data['brand_name'].dropna().unique() 
        if 'asco' in str(c).lower()]
for v in asco:
    print(f'  {v}: {(data["brand_name"] == v).sum():,} rows')

All Parker variants currently in data:

  Parker (90,961 rows)
  Categories: Air Cylinders, Roller Rail Guides, Air Filters

  Parker Dayco (210 rows)
  Categories: General Purpose Hoses, Water Hoses, LP Gas Hoses

  Parker Fluid Connectors (52 rows)
  Categories: General Purpose Hoses, Hoses, Garden Hoses

  Parker Parflex (1,447 rows)
  Categories: Crimp Hose Fittings, Plastic Tubing, Thermoplastic Hoses

  Parker Transair (3 rows)
  Categories: 

  Parker-Commercial Intertech (1 rows)
  Categories: Gear Pumps

  Parker-Pioneer (28 rows)
  Categories: Quick Coupler Nipples, Quick Couplers, Quick Coupler Adapters


Humphrey variants:
  Humphrey Products: 3,706 rows

Asco variants:
  Asco Numatics: 1 rows


In [15]:
# Add to BRAND_ALIASES and re-run normalization
additional_aliases = {
    'parker transair':             'Parker',  # 3 rows, no category
    'parker-commercial intertech': 'Parker',  # 1 row, gear pumps
}

BRAND_ALIASES.update(additional_aliases)

# Re-apply normalization
data['brand_name'] = data['brand_name_raw'].apply(normalize_brand).astype('string')

print('Updated brand counts:')
print(data['brand_name'].value_counts(dropna=False).to_string())

Updated brand counts:
brand_name
Parker                                 90965
Bosch Rexroth                          37420
Versa Products                         22880
Hydac                                  22877
Aventics                               15087
Hengli                                 15052
Balluff                                14543
Schroeder Industries                    9491
Daman Products                          7253
Graco                                   7065
Hengst Filtration                       7063
Phoenix Contact                         6975
Bijur Delimon                           6290
Donaldson                               6142
NOSHOK, Inc.                            6058
Adaptall Inc                            4905
Humphrey Products                       3706
Mitsubishi                              2092
Parker Parflex                          1447
WEG                                      913
Holmbury                                 632
Continental           

In [16]:
# --- Brand update ---
additional_aliases = {
    'parker transair':             'Parker',
    'parker-commercial intertech': 'Parker',
}
BRAND_ALIASES.update(additional_aliases)
data['brand_name'] = data['brand_name_raw'].apply(normalize_brand).astype('string')

print('Brand update done.')
print(f'Unique brands now: {data["brand_name"].nunique()}')

# --- Supplier normalization audit ---
print('\n\nAll supplier_name values + row counts:')
print(data['supplier_name'].value_counts(dropna=False).to_string())

Brand update done.
Unique brands now: 53


All supplier_name values + row counts:
supplier_name
PARKER PNEUMATIC                  70800
VERSA PRODUCTS CO.                22880
HYCON DIV. HYDAC TECHNOLOGY       22878
BOSCH REXROTH INDUSTRIAL          20915
ASCO LP                           15091
HENGLI AMERICA CORPORATION        15052
BALLUFF, INC.                     14543
SCHROEDER INDUSTRIES               9489
DAMAN PRODUCTS CO.                 7253
HENGST FILTRATION                  7089
GRACO AUTO LUBE DIVISION           7065
RS AMERICA                         6974
BIJUR DELIMON INTL.                6290
DONALDSON                          6142
NOSHOK INC.                        6058
PARKER HANNIFIN TUBE FITTINGS      5550
BOSCH REXROTH CONTROLS             5374
BOSCH REXROTH-LT                   5364
ADAPTALL INC                       4905
PARKER HANNIFIN HOSE               4253
HUMPHREY PRODUCTS CO.              3706
PARKER GAS SEPARATION AND FIL      3417
PARKER HANNIFIN Q.C.    

In [17]:
noise_suppliers = ['CREDIT CARD', 'S3', 'LIVINGSTON & HAVEN, LLC']
data['supplier_flag'] = data['supplier_name'].isin(noise_suppliers)
print(f'Noise supplier rows flagged: {data["supplier_flag"].sum()}')

Noise supplier rows flagged: 3


In [18]:
data.to_csv('data/cleaned_data.csv', index=False)
print(f'Saved → data/cleaned_data.csv')
print(f'Rows:    {len(data):,}')
print(f'Columns: {len(data.columns)}')
print(f'\nColumns:')
for c in data.columns:
    filled = data[c].notna().sum()
    pct = filled / len(data) * 100
    print(f'  {c:<40} {pct:5.1f}% filled')

Saved → data/cleaned_data.csv
Rows:    292,640
Columns: 22

Columns:
  supplier_catalog_key                     100.0% filled
  supplier_name                            100.0% filled
  sku                                      100.0% filled
  attribute_family                         100.0% filled
  category_name                             95.1% filled
  category_id                               95.1% filled
  brand_name                                99.9% filled
  manufacturer_part_number                 100.0% filled
  last_sold_price                           99.9% filled
  item_weight                              100.0% filled
  downloads                                 62.2% filled
  manufacturer_description                  13.0% filled
  default_name                             100.0% filled
  livhaven_name                             21.0% filled
  default_short_description                 84.9% filled
  livhaven_short_description                19.2% filled
  mro_description  

In [19]:
import re
from html.parser import HTMLParser
from html import unescape

class _Strip(HTMLParser):
    def __init__(self):
        super().__init__()
        self.fed = []
    def handle_data(self, d):
        self.fed.append(d)
    def get_text(self):
        return ' '.join(self.fed)

def strip_html(t):
    if pd.isna(t) or str(t).strip() == '': return pd.NA
    t = str(t)
    if '<' not in t and '&' not in t: return t.strip()
    # Strip tags
    if '<' in t:
        s = _Strip()
        s.feed(t)
        t = re.sub(r'\s+', ' ', s.get_text()).strip()
    # Decode HTML entities (&amp; → &, &quot; → ", etc.)
    t = unescape(t)
    return t.strip() or pd.NA

# Re-run on ALL text columns including downloads
cols_to_strip = [
    'default_name',
    'default_short_description',
    'livhaven_short_description',
    'mro_description',
    'livhaven_description',
    'manufacturer_description',
    'livhaven_name',
    'downloads',
]

for col in cols_to_strip:
    if col not in data.columns: continue
    data[col] = data[col].apply(strip_html).astype('string')
    remaining = data[col].apply(lambda x: bool(re.search(r'<[a-zA-Z]|&[a-z]+;', str(x))) if pd.notna(x) else False).sum()
    print(f'  {col:<35} HTML/entities remaining: {remaining}')

  default_name                        HTML/entities remaining: 0
  default_short_description           HTML/entities remaining: 53
  livhaven_short_description          HTML/entities remaining: 0
  mro_description                     HTML/entities remaining: 1
  livhaven_description                HTML/entities remaining: 1
  manufacturer_description            HTML/entities remaining: 0
  livhaven_name                       HTML/entities remaining: 0
  downloads                           HTML/entities remaining: 0


In [20]:
# Inspect the remaining ones
for col in ['default_short_description', 'mro_description', 'livhaven_description']:
    mask = data[col].apply(lambda x: bool(re.search(r'<[a-zA-Z]|&[a-z]+;', str(x))) if pd.notna(x) else False)
    print(f'\n=== {col} ({mask.sum()} remaining) ===')
    for val in data.loc[mask, col].head(5):
        print(f'  → {str(val)[:300]}')


=== default_short_description (53 remaining) ===
  → Pressure transducer with analog output 0.1 &helip; 10 V, pressure range 0 ... 630 bar, connector 4-pole M12x1 A-coded, throttle element
  → Pressure transducer with analog output 4 &helip; 20 mA, pressure range 0 &helip; 10 bar, connector 4-pole M12x1 A-coded, throttle element
  → Pressure transducer with analog output 4 &helip; 20 mA, pressure range 0 &helip; 630 bar, connector 4-pole M12x1 A-coded, throttle element
  → Electronic pressure switch, 2 switching outputs and IO-Link, pressure range 0 &helip; 250 bar, connector 4-pole M12x1 A-coded, throttle element
  → Electronic pressure switch, 2 switching outputs and IO-Link, pressure range 0 &helip; 100 bar, connector 4-pole M12x1 A-coded, throttle element

=== mro_description (1 remaining) ===
  → 1J643-8-8 Parker # 8 Female Seal-Lok x 1/2" i.d. Hose Fitting MROStop is an authorized Parker distributor MROStop has been an official distributor of Parker hose and fittings online sinc

In [ ]:
from html import unescape
import re

# Extended entity map for entities unescape() doesn't know
EXTRA_ENTITIES = {
    '&helip;': '...',
    '&mldr;': '...',
    '&hellip;': '...',  # standard ellipsis - belt and suspenders
    '&lsquo;': "'",
    '&rsquo;': "'",
    '&ldquo;': '"',
    '&rdquo;': '"',
    '&ndash;': '-',
    '&mdash;': '--',
    '&nbsp;': ' ',
}

def strip_html_v2(t):
    if pd.isna(t) or str(t).strip() == '': return pd.NA
    t = str(t)
    # Replace extra entities first
    for entity, replacement in EXTRA_ENTITIES.items():
        t = t.replace(entity, replacement)
    # Strip HTML tags
    if '<' in t:
        s = _Strip()
        s.feed(t)
        t = re.sub(r'\s+', ' ', s.get_text()).strip()
    # Standard entity decode
    t = unescape(t)
    return t.strip() or pd.NA

cols_to_strip = [
    'default_name', 'default_short_description', 'livhaven_short_description',
    'mro_description', 'livhaven_description', 'manufacturer_description',
    'livhaven_name', 'downloads',
]

for col in cols_to_strip:
    if col not in data.columns: continue
    data[col] = data[col].apply(strip_html_v2).astype('string')

# only flag actual HTML tags, not curly quotes
for col in cols_to_strip:
    if col not in data.columns: continue
    remaining = data[col].apply(
        lambda x: bool(re.search(r'<[a-zA-Z]|&[a-z]+;', str(x))) if pd.notna(x) else False
    ).sum()
    print(f'  {col:<35} remaining: {remaining}')

  default_name                        remaining: 0
  default_short_description           remaining: 0
  livhaven_short_description          remaining: 0
  mro_description                     remaining: 1
  livhaven_description                remaining: 1
  manufacturer_description            remaining: 0
  livhaven_name                       remaining: 0
  downloads                           remaining: 0


In [22]:
for col in ['mro_description', 'livhaven_description']:
    mask = data[col].apply(
        lambda x: bool(re.search(r'<[a-zA-Z]|&[a-z]+;', str(x))) if pd.notna(x) else False
    )
    for val in data.loc[mask, col].head(2):
        print(f'{col}: {str(val)[:150]}')

mro_description: 1J643-8-8 Parker # 8 Female Seal-Lok x 1/2" i.d. Hose Fitting MROStop is an authorized Parker distributor MROStop has been an official distributor of 
livhaven_description: 1J643-8-8 Parker # 8 Female Seal-Lok x 1/2" i.d. Hose Fitting Livingston & Haven is an official Parker Hannfin distributor, If you have any questions 


In [ ]:
data.to_csv('data/cleaned_data.csv', index=False)
print(f'Saved → data/cleaned_data.csv')
print(f'Rows:    {len(data):,}')
print(f'Columns: {len(data.columns)}')
print(f'\nFinal columns:')
for c in data.columns:
    filled = data[c].notna().sum()
    pct = filled / len(data) * 100
    print(f'  {c:<40} {pct:5.1f}% filled')

Saved → data/cleaned_data.csv
Rows:    292,640
Columns: 22

Final columns:
  supplier_catalog_key                     100.0% filled
  supplier_name                            100.0% filled
  sku                                      100.0% filled
  attribute_family                         100.0% filled
  category_name                             95.1% filled
  category_id                               95.1% filled
  brand_name                                99.9% filled
  manufacturer_part_number                 100.0% filled
  last_sold_price                           99.9% filled
  item_weight                              100.0% filled
  downloads                                 62.2% filled
  manufacturer_description                  13.0% filled
  default_name                             100.0% filled
  livhaven_name                             21.0% filled
  default_short_description                 84.9% filled
  livhaven_short_description                19.2% filled
  mro_descrip

: 